# MCBOMs worked exampleSingle-segment safety-benefit derivation. Manual calculation cross-checked against the optimizer.

In [ ]:
!pip install --quiet 'git+https://github.com/ttitamu/mcboms-optimization.git#egg=mcboms[pulp]'

In [ ]:
# 2-mile rural two-lane segment, 20-year analysis at 7% discount rate.segment_length_miles = 2.0analysis_horizon_years = 20discount_rate = 0.07# Observed crash history, by KABCO severity, annualized.crashes_per_year = {    "K": 0.05,   # Fatal    "A": 0.20,   # Incapacitating injury    "B": 0.40,   # Non-incapacitating injury    "C": 0.60,   # Possible injury    "O": 1.00,   # Property damage only}# USDOT BCA Guidance May 2025 unit costs (2024 dollars), per crash.crash_cost_by_severity = {    "K": 13_700_000,    "A":    843_000,    "B":    278_000,    "C":    176_000,    "O":     14_700,}# Treatment: centerline rumble strips.cmf_total_crashes = 0.85    # 15% reduction (FHWA CMF Clearinghouse)treatment_cost   = 30_000   # $/mile

In [ ]:
# Annual safety benefit = sum over severities of (crashes * cost) * (1 - CMF).annual_crash_cost = sum(    crashes_per_year[k] * crash_cost_by_severity[k] for k in crashes_per_year)annual_safety_benefit = annual_crash_cost * (1 - cmf_total_crashes)# Convert to present value using uniform-series present-worth factor.def uspw_factor(rate, years):    return ((1 + rate) ** years - 1) / (rate * (1 + rate) ** years)pwf = uspw_factor(discount_rate, analysis_horizon_years)pv_safety_benefit = annual_safety_benefit * pwfpv_treatment_cost = treatment_cost * segment_length_milesprint(f"Annual safety benefit:  ${annual_safety_benefit:>15,.0f}")print(f"Present-value benefit:  ${pv_safety_benefit:>15,.0f}")print(f"Treatment cost:         ${pv_treatment_cost:>15,.0f}")print(f"Net benefit (manual):   ${pv_safety_benefit - pv_treatment_cost:>15,.0f}")

**Run through the optimizer.** A single-segment problem is trivial by hand, but the optimizer should reproduce the manual result.

In [ ]:
import pandas as pdfrom mcboms.core.optimizer import Optimizersites = pd.DataFrame({    "site_id": [1],    "facility_type": ["rural_2lane"],})alternatives = pd.DataFrame({    "site_id":      [1, 1],    "alt_id":       [0, 1],    "description":  ["do_nothing", "centerline_rumble_strips"],    "total_cost":   [0, pv_treatment_cost],    "total_benefit":[0, pv_safety_benefit],})opt = Optimizer(sites=sites, alternatives=alternatives, budget=1_000_000)result = opt.solve()print(result.summary())

In [ ]:
# Manual derivation and optimizer should agree to the cent.manual_net = pv_safety_benefit - pv_treatment_costassert abs(manual_net - result.net_benefit) < 1.0print(f"Manual:    ${manual_net:>15,.0f}")print(f"Optimizer: ${result.net_benefit:>15,.0f}")